# Bar Data

Adapted from [ib_async bar_data](https://ib-api-reloaded.github.io/ib_async/notebooks.html).
Demonstrates head timestamp and historical bar requests.

In [ ]:
import os, threading, time
from dotenv import load_dotenv
from ibx import EWrapper, EClient, Contract, BarData

load_dotenv()
USERNAME = os.environ["IB_USERNAME"]
PASSWORD = os.environ["IB_PASSWORD"]

In [ ]:
class App(EWrapper):
    def __init__(self):
        super().__init__()
        self.bars = {}          # req_id -> list[BarData]
        self.head_ts = {}       # req_id -> str
        self.done = {}          # req_id -> Event
        self.connected = threading.Event()

    def next_valid_id(self, order_id):
        self.connected.set()

    def head_timestamp(self, req_id, head_timestamp):
        self.head_ts[req_id] = head_timestamp
        if req_id in self.done:
            self.done[req_id].set()

    def historical_data(self, req_id, bar):
        self.bars.setdefault(req_id, []).append(bar)

    def historical_data_end(self, req_id, start, end):
        if req_id in self.done:
            self.done[req_id].set()

    def error(self, req_id, error_code, error_string, advanced_order_reject_json=""):
        if error_code not in (2104, 2106, 2158):
            print(f"Error {error_code}: {error_string}")

In [ ]:
app = App()
client = EClient(app)
client.connect(username=USERNAME, password=PASSWORD, paper=True)

thread = threading.Thread(target=client.run, daemon=True)
thread.start()
app.connected.wait(timeout=10)
print(f"Connected: {client.is_connected()}")

### Head Timestamp

Get the earliest available bar data date:

In [ ]:
contract = Contract(con_id=265598, symbol="AAPL")

app.done[1] = threading.Event()
client.req_head_time_stamp(1, contract, "TRADES", 1)
app.done[1].wait(timeout=10)

print(f"Earliest data: {app.head_ts.get(1, 'no response')}")

### Historical Bars

Request hourly bars for the last 5 trading days:

In [ ]:
app.done[2] = threading.Event()
app.bars[2] = []

client.req_historical_data(
    2, contract,
    end_date_time="",
    duration_str="5 D",
    bar_size_setting="1 hour",
    what_to_show="TRADES",
    use_rth=1,
)
app.done[2].wait(timeout=15)

bars = app.bars.get(2, [])
print(f"Got {len(bars)} bars")
if bars:
    print(f"  First: {bars[0].date}  O={bars[0].open}  H={bars[0].high}  L={bars[0].low}  C={bars[0].close}  V={bars[0].volume}")
    print(f"  Last:  {bars[-1].date}  O={bars[-1].open}  H={bars[-1].high}  L={bars[-1].low}  C={bars[-1].close}  V={bars[-1].volume}")

### Plot the close prices

In [ ]:
import pandas as pd

if bars:
    df = pd.DataFrame([
        {"date": b.date, "open": b.open, "high": b.high,
         "low": b.low, "close": b.close, "volume": b.volume}
        for b in bars
    ])
    display(df.head())
    display(df.tail())

In [ ]:
%matplotlib inline

if bars:
    df.plot(y="close", title="AAPL Close (1h bars)", figsize=(12, 4))

In [ ]:
client.disconnect()